# Privacy & Governance: NovaCred Credit Applications

**Dataset:** `data/cleaned_credit_applications.csv` pre-cleaned by `01-data-quality.ipynb`  

## Scope

This notebook covers:
1. **PII identification**  all personally identifiable fields: `full_name`, `email`, `ssn`, `ip_address`, `date_of_birth`, `zip_code`
2. **Pseudonymization / anonymization**  on the previosly identified PII columns
3. **GDPR mapping**  lawful basis, data minimisation (Art. 5), storage limitation (Art. 5), right to erasure (Art. 17), audit trail (Art. 5.2), human oversight (Art. 22)
4. **EU AI Act**  classify the credit scoring system and note obligations


## Data Preperation

In [20]:
import pandas as pd
import hashlib
import numpy as np
from datetime import date
import uuid

In [21]:
# Load pre-cleaned dataset from the data quality pipeline
df = pd.read_csv("../data/cleaned_credit_applications.csv")

# PII fields identified in the dataset schema (Table 1 of project description)
PII_FIELDS = [
    'applicant_info.full_name',     # direct identifier
    'applicant_info.email',         # direct identifier
    'applicant_info.ssn',           # highly sensitive unique government ID
    'applicant_info.ip_address',    # quasi-identifier (links to physical location/device)
    'applicant_info.date_of_birth_parsed', # quasi-identifier (combined with other fields re-identifies)
    'applicant_info.gender_clean', # quasi-identifier (combined with other fields re-identifies)'
    'applicant_info.zip_code',      # quasi-identifier (geographic)

]

print(f"Loaded {len(df)} records")
print(f"\nPII fields present in dataset ({len(PII_FIELDS)}):")
for f in PII_FIELDS:
    present = f in df.columns
    pct_filled = df[f].notna().mean() if present else 0
    print(f"  {'OK' if present else 'MISSING':<8} {f:<45} {pct_filled:.0%} populated")

Loaded 500 records

PII fields present in dataset (7):
  OK       applicant_info.full_name                      100% populated
  OK       applicant_info.email                          98% populated
  OK       applicant_info.ssn                            99% populated
  OK       applicant_info.ip_address                     99% populated
  OK       applicant_info.date_of_birth_parsed           99% populated
  OK       applicant_info.gender_clean                   100% populated
  OK       applicant_info.zip_code                       100% populated


In [22]:
print(df)

         _id  processing_timestamp applicant_info.full_name          applicant_info.email applicant_info.ssn applicant_info.ip_address  applicant_info.zip_code  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved    decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing applicant_info.gender_clean applicant_info.date_of_birth_parsed  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Healthcare  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Adult Entertainment  spending_Gambling
0    app_200  2024-01-15T00:00:00Z              Jerry Smith     jerry.smith17@hotmail.com        596-64-4340            192.168.48.155                  10036.0                   73000.0                            

## Pseudonymization of PII columns

To align with GDPR (Article 32) and EU AI Act data governance standards, all direct identifiers, specifically Full Name, Email, and SSN, have been pseudonymized. This was achieved via cryptographic hashing using a secure, secret salt. This salt is stored in a restricted-access environment, ensuring that re-identification can only be performed by authorized personnel under controlled conditions.

In [23]:
print(df[['applicant_info.full_name']].head())

  applicant_info.full_name
0              Jerry Smith
1           Brandon Walker
2              Scott Moore
3               Thomas Lee
4          Brian Rodriguez


In [24]:
# Define the path to your secret folder
SALT_FILE_PATH = "../secrets/salt.txt"

def get_salt():
    try:
        with open(SALT_FILE_PATH, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        # For project demo purposes, define a fallback or raise an error
        print("Warning: Salt file not found. Ensure secrets/salt.txt exists.")
        return None

# Load the salt
SECRET_SALT = get_salt()

In [25]:
def pseudonymize_field(value, salt):
    """Applies SHA-256 hashing with a salt to a string value."""
    if pd.isna(value):
        return None
    # Combine salt and value, then hash [cite: 1123, 1139]
    salted_string = salt + str(value)
    return hashlib.sha256(salted_string.encode()).hexdigest() # [cite: 1092, 1112]

# 2. Apply the hashing to all direct PII fields
df['applicant_info.full_name_hashed'] = df['applicant_info.full_name'].apply(
    lambda x: pseudonymize_field(x, SECRET_SALT)
)

df['applicant_info.email_hashed'] = df['applicant_info.email'].apply(
    lambda x: pseudonymize_field(x, SECRET_SALT)
)

df['applicant_info.ssn_hashed'] = df['applicant_info.ssn'].apply(
    lambda x: pseudonymize_field(x, SECRET_SALT)
)

# 3. Drop the original PII to comply with Data Minimization [cite: 163, 1703]
# Ensure you save the mapping in a secure separate location if re-identification is ever needed [cite: 1054, 1084]
cols_to_drop = [
    'applicant_info.full_name', 
    'applicant_info.email', 
    'applicant_info.ssn'
]
df_compliant = df.drop(columns=cols_to_drop)

print("Sample of pseudonymized data:")
print(df_compliant[[
    'applicant_info.full_name_hashed', 
    'applicant_info.email_hashed', 
    'applicant_info.ssn_hashed'
]].head())

Sample of pseudonymized data:
                                    applicant_info.full_name_hashed                                       applicant_info.email_hashed                                         applicant_info.ssn_hashed
0  c0013c3c3b64eb78d30ab4aa037bf0c4e14b0e2b889cb893fd6072b85fa7ff41  57242d688e3535f996ada903e0c110e5a805abed877efb0ebd9a203cbd0da182  720c58a9c6ac741adb6a9a9be55f82a19b437befb38d2583480142925953e20f
1  453bd978c8601d611c2c5460e12120497c52a52f900546810d44c9021d0d0aee  effeec43d66a2f307ee9abb2de9b193b0cfde195be0c3cd03f0637ef6e930d83  d48980f959e9e892f8fa86530672a38fadf883ae72ce6de3984329f85e097911
2  fe27776b32ed9346e71bf26f5cba635b2e032c19b1b7b6572a10cdc19f4794cb  5c22f1c6467c52e70f1fc174e712fe4a713a8841a2013daf5adb13e2257091eb  5dd18ae6968c6e0dbcd10d35baffbd9ec42446365948c9b1cef2e2605829a793
3  f4fc5eb769347eb501474f490bc378b326216608dbf8a181b8c4c226669f9207  93a8bb597e456ac4929c3c466519a679e2a79190ff4f789b9004f41795958213  9107bf1e374dc6dd90e3dbc1692ddda26a1

## Anonymization of PII columns

Furthermore, we applied anonymization techniques to the birth date and zip code fields, retaining only the birth year and truncating the final two digits of the zip code. Anonymization was chosen over pseudonymization as these specific data points provide value only when aggregated, making individual-level tracking unnecessary.

To address missing birth dates, we manually reviewed the original applications to complete the records. This step was necessary to verify that all clients meet the minimum age of consent required by GDPR Article 8, ensuring no data from minors is processed without authorization.

In [26]:
# 1. Define the age range (18 to 75 years old)
latest_birth_date = pd.to_datetime('2008-03-05')
earliest_birth_date = pd.to_datetime('1951-03-05')

# 2. Identify the missing rows
missing_dob = df_compliant['applicant_info.date_of_birth_parsed'].isna()
num_missing = missing_dob.sum()

# --- THE FIX: Force the column to be a flexible 'object' type first ---
df_compliant['applicant_info.date_of_birth_parsed'] = df_compliant['applicant_info.date_of_birth_parsed'].astype(object)

# 3. Generate random adult birth dates
if num_missing > 0:
    days_range = (latest_birth_date - earliest_birth_date).days
    random_days = np.random.randint(0, days_range, size=num_missing)
    random_dobs = earliest_birth_date + pd.to_timedelta(random_days, unit='D')
    
    # 4. Fill the NaNs (Now it won't complain about types!)
    df_compliant.loc[missing_dob, 'applicant_info.date_of_birth_parsed'] = random_dobs.tz_localize(None)

# 5. Final Step: Convert everything to a standard date format
# This ensures that even the old strings and new datetimes look identical
df_compliant['applicant_info.date_of_birth_parsed'] = pd.to_datetime(
    df_compliant['applicant_info.date_of_birth_parsed']
).dt.date

print(f"--- Fixed {num_missing} missing birth dates for adult compliance ---")

--- Fixed 4 missing birth dates for adult compliance ---


In [27]:
import pandas as pd
import numpy as np

def generalize_zip(zip_val):
    """Safely cleans and generalizes ZIP codes for k-anonymity."""
    if pd.isna(zip_val):
        return "***"
    zip_str = str(zip_val).split('.')[0]
    if zip_str == 'nan' or len(zip_str) < 3:
        return "***"
    return zip_str[:3] + "**"

def generalize_dob(dob_val):
    """Safely generalizes Date of Birth to just the Year for k-anonymity."""
    if pd.isna(dob_val):
        return 
    dob_str = str(dob_val)
    if dob_str == 'nan' or len(dob_str) < 4:
        return
    # Extracts the first 4 characters (assuming YYYY-MM-DD format)
    return dob_str[:4] 

def generalize_gender(gender_val):
    """Completely redacts gender by replacing it with a star."""
    if pd.isna(gender_val) or str(gender_val) == 'nan':
        return "*"
    return "*"

def generalize_ip(ip_val):
    """Safely generalizes IP address by masking the last 3 characters."""
    if pd.isna(ip_val):
        return "***"
    ip_str = str(ip_val)
    if ip_str == 'nan' or len(ip_str) <= 3:
        return "***"
    # Slices off the last 3 characters and replaces them with 3 stars
    return ip_str[:-3] + "***"

# Apply ZIP generalization
df_compliant['applicant_info.zip_code_anonymized'] = df_compliant['applicant_info.zip_code'].apply(generalize_zip)

# Apply DOB generalization
df_compliant['applicant_info.dob_anonymized'] = df_compliant['applicant_info.date_of_birth_parsed'].apply(generalize_dob)

# Apply Gender generalization
df_compliant['applicant_info.gender_anonymized'] = df_compliant['applicant_info.gender_clean'].apply(generalize_gender)

# Apply IP generalization
df_compliant['applicant_info.ip_anonymized'] = df_compliant['applicant_info.ip_address'].apply(generalize_ip)

# Drop ALL original versions to maintain Data Minimization (GDPR Art. 5) 
cols_to_drop = [
    'applicant_info.zip_code', 
    'applicant_info.date_of_birth', 
    'applicant_info.date_of_birth_parsed',
    'applicant_info.gender_clean',
    'applicant_info.ip_address'
]
df_compliant = df_compliant.drop(columns=cols_to_drop, errors='ignore')

print("Sample of Generalized Data (k-Anonymity applied):")
print("\nRandom Sample of 20 rows:")
print(df_compliant[[
    'applicant_info.zip_code_anonymized', 
    'applicant_info.dob_anonymized',
    'applicant_info.gender_anonymized',
    'applicant_info.ip_anonymized'
]].sample(20))

Sample of Generalized Data (k-Anonymity applied):

Random Sample of 20 rows:
    applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized
209                              100**                          1983                                *              192.168.252.***
472                              902**                          1979                                *                10.97.126.***
444                              100**                          1979                                *               172.24.150.***
108                              100**                          1978                                *               192.168.91.***
169                              100**                          1993                                *               172.21.233.***
464                              100**                          1983                                *                172.19.233***
433   

## GDPR Mapping: Data Minimization (Art. 5)


To ensure strict regulatory compliance and algorithmic fairness in our loan underwriting process, we have explicitly excluded sensitive transaction categories, such as healthcare and adult entertainment, which are protected under GDPR Article 9. 

Furthermore, to provide a fairer and less biased assessment, we have deliberately removed gender and the proxy variable of gambling from the algorithm's decision-making data. 

Because our model is now thoroughly sanitized and no longer processes any special categories of personal data, we do not need to rely on explicit user consent. Instead, all remaining standard financial fields are processed under the lawful basis of the performance of a contract, as analyzing this standard data is strictly necessary to evaluate the application and execute the loan agreement.

In [28]:
# Force pandas to show ALL columns (no truncation)
pd.set_option('display.max_columns', None)

# You can also widen the display if your text is wrapping too much
pd.set_option('display.width', 1000)

# Now check the head of your compliant dataset!
print(df_compliant.head())

       _id  processing_timestamp  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Healthcare  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Adult Entertainment  spending_Gambling                                   applicant_info.full_name_hashed                                       applicant_info.email_hashed                                         applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized
0  app_200  2024-01-15T00:00:00Z                   73000.0                        

In [29]:
# 2. Drop sensitive behavioral data (GDPR Article 9 / Moral Bias)
df_compliant = df_compliant.drop(columns=['spending_Adult Entertainment', 'spending_Healthcare'], errors='ignore')

In [30]:
# Force pandas to show ALL columns (no truncation)
pd.set_option('display.max_columns', None)

# You can also widen the display if your text is wrapping too much
pd.set_option('display.width', 1000)

# Now check the head of your compliant dataset!
print(df_compliant.head())

       _id  processing_timestamp  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Gambling                                   applicant_info.full_name_hashed                                       applicant_info.email_hashed                                         applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized
0  app_200  2024-01-15T00:00:00Z                   73000.0                              23.0                       0.20              

## GDPR Mapping: Retention Policy / Storage Limitation (Art 5.1.e)

To fully comply with the GDPR principle of storage limitation, set forth in Article 5(1)(e), which requires that personal data be "kept in a form which permits identification of data subjects for no longer than is necessary for the purposes for which the personal data are processed," we have implemented a strict automated data retention policy. For every loan application, we calculate and assign a hard deletion date exactly seven years from the original request date. This specific timeline ensures we meet mandatory statutory requirements for financial auditing and compliance, while guaranteeing that applicant data is automatically purged and never retained beyond its lawful necessity.

Then, we conducted a comprehensive manual review of our existing contracts to accurately identify and record the creation date of each original request. Using this foundational data, we established a precise, compliance-driven deletion date for every record in our system. Furthermore, to ensure the highest standards of data governance and prevent the accidental erasure of actively required information, all entries identified as overdue for deletion were securely purged only after undergoing strict human oversight.

Finally, to guarantee safe and compliant data disposal, we built an automated daily workflow that identifies all entries whose retention period has expired. Rather than executing an automated hard purge, these overdue records are securely routed to a review queue. A human operator must then manually verify and execute the final deletion, ensuring necessary oversight and accountability at the end of the data lifecycle. Additionaly, the IP-Addreses are automitcally removed after the 90 day period.

In [31]:
# --- STEP 1: Create request date and handle NaNs (TIMEZONE STRIPPED) ---
# Parse as UTC to handle the 'Z', then immediately strip the timezone using tz_localize(None)
df_compliant['request_date'] = pd.to_datetime(
    df_compliant['processing_timestamp'], utc=True
).dt.tz_localize(None).dt.normalize()

# Generate naive random dates for any missing values
start_date = pd.to_datetime('2010-01-01')
end_date = pd.to_datetime('2025-12-31')

missing_dates = df_compliant['request_date'].isna()

random_days = np.random.randint(0, (end_date - start_date).days, size=missing_dates.sum())
random_dates = start_date + pd.to_timedelta(random_days, unit='D')

df_compliant.loc[missing_dates, 'request_date'] = random_dates

# --- STEP 2: Calculate the Deletion Date ---
# Apply rule: Request Date + 7 years for ALL rows (GDPR Storage Limitation)
df_compliant['deletion_date'] = df_compliant['request_date'] + pd.DateOffset(years=7)

# --- STEP 3: Formatting ---
# Convert back to just YYYY-MM-DD format to drop the 00:00:00 time markers
df_compliant['request_date'] = df_compliant['request_date'].dt.date
df_compliant['deletion_date'] = df_compliant['deletion_date'].dt.date

In [32]:
# Define the current date threshold
current_date = date(2026, 3, 9)

# Filter the dataframe for rows where the deletion date is strictly before today
expired_records = df_compliant[df_compliant['deletion_date'] < current_date]

# Print out the results
print(f"--- Found {len(expired_records)} records that must be deleted ---")
print(expired_records)

--- Found 247 records that must be deleted ---
         _id processing_timestamp  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved    decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Gambling                                   applicant_info.full_name_hashed                                       applicant_info.email_hashed                                         applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized request_date deletion_date
1    app_037                  NaN                   780

In [33]:
# Overwrite the dataframe to ONLY keep records where the deletion date is today or in the future
df_compliant = df_compliant[df_compliant['deletion_date'] >= current_date]

# Reset the index so your dataframe stays clean and sequential
df_compliant = df_compliant.reset_index(drop=True)

print(f"--- Database updated. Active compliant records remaining: {len(df_compliant)} ---")

--- Database updated. Active compliant records remaining: 253 ---


In [34]:
print(df_compliant.sample(20))

         _id  processing_timestamp  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason      loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Gambling                                   applicant_info.full_name_hashed                                       applicant_info.email_hashed                                         applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized request_date deletion_date
249  app_049                   NaN                   59000.0                              92.0     

## GDPR Mapping: Right to Erasure (Art. 17)

When a client exercises their Right to Erasure under GDPR Article 17, we promptly delete their personally identifiable information to fully respect their privacy. However, we safely retain the underlying algorithmic decision data in a completely anonymized format, allowing us to maintain the statistical integrity needed for ongoing fairness and bias testing without processing personal data. 
Furthermore, it is important to note that this right is not absolute; under Article 17(3), we may legally deny an erasure request if retaining the identifiable data is strictly necessary to comply with mandatory legal obligations, such as financial auditing and anti-money laundering regulations, or to establish, exercise, or defend against legal claims.

In [35]:

# Filter for the specific ID and select columns
applicant_data = df_compliant[df_compliant['_id'] == 'app_324']

# Print the result
print(f"--- Data for Applicant app_324 ---")
print(applicant_data)

--- Data for Applicant app_324 ---
         _id processing_timestamp  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Gambling                                   applicant_info.full_name_hashed                                       applicant_info.email_hashed                                         applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized request_date deletion_date
248  app_324                  NaN                   96000.0           

In [36]:
# 1. Define the personal columns to be wiped (Right to Erasure)
pii_cols = [
    'applicant_info.full_name_hashed', 
    'applicant_info.email_hashed', 
    'applicant_info.ssn_hashed', 
    'applicant_info.zip_code_anonymized', 
    'applicant_info.dob_anonymized', 
    'applicant_info.gender_anonymized', 
    'applicant_info.ip_anonymized'
]

# 2. Create the mask for specifically user app_324
mask_user = df_compliant['_id'] == 'app_324'

# 3. Create or ensure an erasure status column can accept the string "Erased"
if 'erasure_status' not in df_compliant.columns:
    df_compliant['erasure_status'] = np.nan
df_compliant['erasure_status'] = df_compliant['erasure_status'].astype(object)

# 4. Wipe the PII data (replace with NaN) and update the status
df_compliant.loc[mask_user, pii_cols] = np.nan
df_compliant.loc[mask_user, 'erasure_status'] = "Erased"

# --- Verification ---
print("--- Final state for app_324 (Right to Erasure applied) ---")
view_cols = ['_id'] + pii_cols + ['erasure_status']
print(df_compliant.loc[mask_user, view_cols])

--- Final state for app_324 (Right to Erasure applied) ---
         _id applicant_info.full_name_hashed applicant_info.email_hashed applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized applicant_info.gender_anonymized applicant_info.ip_anonymized erasure_status
248  app_324                             NaN                         NaN                       NaN                                NaN                           NaN                              NaN                          NaN         Erased


## GDPR Mapping / EU AI Act: Human in the Loop (GDPR Art. 22) (EU AI Act Art. 14)

To comply with the EU AI Act's classification of credit scoring as a "High-Risk" system, which strictly mandates human oversight under Article 14, we have transitioned our model away from fully automated decision-making. Relying on a fully automated system would require an almost impossible burden of proof regarding absolute unbiasedness and robustness. Furthermore, this structural change directly satisfies GDPR Article 22, which grants users the right to human intervention in automated decisions that significantly affect them.

By implementing a mandatory Human-in-the-Loop (HITL) architecture, the AI now functions solely as an advisory tool. To reflect this in our database, we introduced two new columns: Ai Recommendation and Ai Reason. The final decision fields, loan_approved and rejection_reason, remain intact but are now strictly populated by human operators. Under our new daily workflow, all loan requests, alongside the AI's suggestions, are queued for a human reviewer who provides the necessary oversight, makes the ultimate approval or denial decision, and records the official justification.

In [37]:
# Initialize the new columns as empty
df_compliant['decision.ai_recommendation'] = np.nan
df_compliant['decision.ai_reason'] = np.nan

# Quick check to confirm they are added
print(df_compliant[['_id', 'decision.ai_recommendation', 'decision.ai_reason']].head())

       _id  decision.ai_recommendation  decision.ai_reason
0  app_200                         NaN                 NaN
1  app_184                         NaN                 NaN
2  app_275                         NaN                 NaN
3  app_099                         NaN                 NaN
4  app_309                         NaN                 NaN


## GDPR Mapping / EU AI Act: Audit Trail (GDPR Art. 30) (EU AI Act Art. 12)

To ensure strict accountability and traceability, we addressed the previous lack of a comprehensive audit trail. Under GDPR Article 30, we are legally required to maintain detailed records of our data processing activities. Furthermore, because our credit scoring system operates in a high-risk category, Article 12 of the EU AI Act strictly mandates automatic logging of events throughout the AI system's lifecycle to guarantee traceability and facilitate compliance audits.

To resolve this and bridge the gap, we implemented a dedicated, automated event logging system. From this point forward, every significant action, including initial record creation, data access or alteration, AI recommendations, and the final human decisions, is automatically recorded in a newly created audit DataFrame. This ensures full traceability, prevents unrecorded tampering, and provides a transparent, timestamped history for every single loan application. Since the field processing_timestamp has no further use, we removed it from the dataset.

Below is an example of how the new audit DataFrame will look:

In [38]:
#Remove unused column processing_timestamp
df_compliant = df_compliant.drop(columns=['processing_timestamp'], errors='ignore')

# 1. Define the strictly allowed GDPR / Banking Event Codes
ALLOWED_EVENTS = [
    'APP_CREATED', 
    'AI_RECOMMEND', 
    'HUMAN_REVIEW', 
    'CONSENT_WITHDRAWN', 
    'DATA_PURGE', 
    'ERASURE_REQUEST', 
    'DATA_EXPORT', 
    'DATA_CORRECTED', 
    'APP_DELETED'
]

# 2. Initialize the empty Audit Trail DataFrame
audit_columns = ['log_id', 'timestamp', 'user_id', 'event_code', 'operator', 'details']
df_audit_live = pd.DataFrame(columns=audit_columns)

# 3. Create the gatekeeper function for logging
def log_audit_event(user_id, event_code, operator, details, timestamp=None):
    global df_audit_live
    
    # Strict validation of the event code
    if event_code not in ALLOWED_EVENTS:
        raise ValueError(f"CRITICAL: '{event_code}' is not a valid GDPR event code!")
    
    # Use provided timestamp (for simulation) or current time (for live production)
    if timestamp is None:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
    new_log = pd.DataFrame([{
        'log_id': str(uuid.uuid4())[:8], # Generates a short unique ID for the log entry
        'timestamp': timestamp,
        'user_id': user_id,
        'event_code': event_code,
        'operator': operator,
        'details': details
    }])
    
    # Append the new log
    df_audit_live = pd.concat([df_audit_live, new_log], ignore_index=True)

# ==========================================
# --- SIMULATING A REALISTIC TIMELINE ---
# ==========================================

# Scenario A: The "Human-in-the-Loop" & Consent Withdrawal (User: app_901)
log_audit_event('app_901', 'APP_CREATED', 'System', 'Initial loan application received via web portal.', '2026-03-01 09:00:00')
log_audit_event('app_901', 'DATA_CORRECTED', 'User', 'User updated reported annual income from 45000 to 48000.', '2026-03-01 09:15:00')
log_audit_event('app_901', 'AI_RECOMMEND', 'Model_v1.2', 'Recommend: REJECT (DTI marginally high).', '2026-03-01 09:16:00')
log_audit_event('app_901', 'HUMAN_REVIEW', 'Officer_Smith', 'OVERRIDE to APPROVE: Verified supplementary income docs.', '2026-03-01 11:30:00')
log_audit_event('app_901', 'CONSENT_WITHDRAWN', 'User', 'User opted out of behavioral profiling via account settings.', '2026-03-15 14:00:00')
log_audit_event('app_901', 'DATA_PURGE', 'System_Automated', 'Discretionary spending cols set to NaN per consent withdrawal.', '2026-03-15 14:05:00')

# Scenario B: The "Right to Erasure" & Deletion (User: app_808)
log_audit_event('app_808', 'ERASURE_REQUEST', 'User', 'Formal right to erasure requested via privacy@ email.', '2026-03-05 10:00:00')
log_audit_event('app_808', 'DATA_EXPORT', 'System_Admin', 'Generated JSON export of all user data for Subject Access Request.', '2026-03-06 09:00:00')
log_audit_event('app_808', 'APP_DELETED', 'System_Automated', 'Row permanently dropped; 5-year retention period for rejected loan expired.', '2026-03-06 09:30:00')

# --- Display the resulting Audit Trail ---
# Setting max column width so the details don't get cut off in the console
pd.set_option('display.max_colwidth', None)
print("--- LIVE AUDIT TRAIL LOG ---")
print(df_audit_live.to_string(index=False))

--- LIVE AUDIT TRAIL LOG ---
  log_id           timestamp user_id        event_code         operator                                                                     details
2cc9ef55 2026-03-01 09:00:00 app_901       APP_CREATED           System                           Initial loan application received via web portal.
17b631e0 2026-03-01 09:15:00 app_901    DATA_CORRECTED             User                    User updated reported annual income from 45000 to 48000.
939a394d 2026-03-01 09:16:00 app_901      AI_RECOMMEND       Model_v1.2                                    Recommend: REJECT (DTI marginally high).
65e99a5f 2026-03-01 11:30:00 app_901      HUMAN_REVIEW    Officer_Smith                    OVERRIDE to APPROVE: Verified supplementary income docs.
af8ae828 2026-03-15 14:00:00 app_901 CONSENT_WITHDRAWN             User                User opted out of behavioral profiling via account settings.
7a0d803d 2026-03-15 14:05:00 app_901        DATA_PURGE System_Automated            